# Complexified PDE: 2D Complex Ginzburg-Landau Equation
## Native Complex Field Simulation — Spatiotemporal Chaos, Spiral Waves, and Pattern Formation

This notebook demonstrates **Approach 1: Native Complex** by simulating the 2D Complex Ginzburg-Landau Equation (CGLE) using the pseudo-spectral `PDESolver` framework.

Instead of splitting the field into two real equations ($u$ and $v$), we define a single complex-valued field $A(x, y, t) = u + iv$. The solver and SymPy handle the complex arithmetic natively, which is mathematically equivalent to a coupled system of two real PDEs but far more concise to write and read.

The CGLE is a paradigmatic model for nonlinear dynamics, describing systems near a Hopf bifurcation, including nonlinear optics, chemical oscillations, and fluid dynamics.

---

## 1. The Governing Equation

$$
\partial_t A = A + (1 + i\alpha)\nabla^2 A - (1 + i\beta)A|A|^2
$$

* $A(x, y, t)$: Complex-valued amplitude field.
* $\alpha, \beta$: Real parameters controlling dispersion and nonlinear frequency shifting.
* The term $A$ represents linear growth.
* The term $(1 + i\alpha)\nabla^2 A$ represents diffusion ($1$) and dispersion ($\alpha$).
* The term $-(1 + i\beta)A|A|^2$ provides nonlinear saturation.

---

## 2. Reformulation for the Solver

We expand the linear operator in Fourier space, where $\nabla^2 \to -k^2 = -(\xi^2 + \eta^2)$:

$$
A + (1 + i\alpha)\nabla^2 A \longrightarrow A - (1 + i\alpha)(\xi^2 + \eta^2)A
$$

The full linear symbol is:

$$
\text{Linear symbol:} \quad 1 - (1 + i\alpha)(\xi^2 + \eta^2)
$$

The equation in the solver's format:

$$
\partial_t A = \underbrace{\text{psiOp}\left(1 - (1 + i\alpha)(\xi^2 + \eta^2), A\right)}_{\text{Linear diffusion and dispersion}} + \underbrace{-(1 + i\beta)A|A|^2}_{\text{Nonlinear saturation}}
$$

---

## 3. Physical Phenomena

* **Plane Waves**: Uniform oscillating states, stable for certain $\alpha, \beta$.
* **Spiral Waves**: Topological defects with rotating phase, common in 2D CGLE.
* **Defect Turbulence**: For large $\alpha, \beta$, the system exhibits chaotic creation and annihilation of phase defects (Benjamin-Feir instability).

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Ginzburg-Landau Coefficients ──
# Examples:
# Spiral wave / defect turbulence regime
ALPHA = 0.5       # Dispersion parameter
BETA = -1.0       # Nonlinear frequency shift parameter
# Defect Turbulence (Benjamin-Feir instability - Chaotic creation and annihilation of phase defects)
# ALPHA = 1.5
# BETA = -1.5
# Plane Waves (stable uniform oscillation - Uniform amplitude, simple phase gradient)
# ALPHA = 0.0
# BETA = 0.0

# ── Grid and Time ──
Lx, Ly = 40.0, 40.0   # Domain size
Nx, Ny = 512, 512     # Resolution
# Nx, Ny = 1024, 1024 # Resolution (higher)

Lt, Nt = 100.0, 2000  # Simulation time and steps
n_frames = 200        # Frames for animation

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')

## 3. SymPy symbols and principal symbol

In [ ]:
t, x, y = sp.symbols('t x y', real=True)
# Using 'xi' and 'eta' for Fourier wavenumbers
xi, eta = sp.symbols('xi eta', real=True)

# Define A as a complex-valued field
A_field = sp.Function('A')(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂A/∂t = A + (1 + i α)∇²A
# Fourier: ∇² → -(ξ² + η²)
# So: 1 - (1 + i α)(ξ² + η²)

k2 = xi**2 + eta**2
symbol_linear = 1 - (1 + sp.I * ALPHA) * k2

print('Principal symbol (linear part):')
print('  a(ξ, η) = ', symbol_linear)

## 4. Complex Ginzburg-Landau equation

In [ ]:
# ∂A/∂t = psiOp(1 - (1 + i α)k², A) - (1 + i β) * A * |A|²

# Nonlinear term: -(1 + i β) * A * |A|²
# In SymPy, |A|² is represented as A * conjugate(A)
nonlinear_term = -(1 + sp.I * BETA) * A_field * sp.conjugate(A_field) * A_field

equation = sp.Eq(
    sp.diff(A_field, t),
    psiOp(symbol_linear, A_field) + nonlinear_term
)

print('Complex Ginzburg-Landau Equation:')
print('  ∂A/∂t = psiOp(1 - (1 + i α)(ξ² + η²), A) - (1 + i β) * A * |A|²')
print(f'\nParameters: α={ALPHA}, β={BETA}')

## 5. Initial conditions

In [ ]:
def initial_condition_cgle(xx, yy):
    """
    Small random complex noise to trigger pattern formation.
    The linear instability amplifies modes, leading to spiral waves
    or defect turbulence depending on α and β.
    """
    np.random.seed(42)
    noise_amplitude = 0.1
    # Complex noise: real and imaginary parts
    noise_real = noise_amplitude * np.random.randn(xx.shape[0], xx.shape[1])
    noise_imag = noise_amplitude * np.random.randn(xx.shape[0], xx.shape[1])
    return noise_real + 1j * noise_imag

print("Using: Random complex noise for CGLE pattern formation")

## 6. Solver setup

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition_cgle,
    n_frames=n_frames,
    plot=True,
)

In [ ]:
solver.psi_ops[0][1].interactive_symbol_analysis(
                            xlim=(-Lx/2, Lx/2),
                            ylim=(-Ly/2, Ly/2),
                            xi_range=(-10, 10),
                            eta_range=(-10, 10),
                            density=50)

## 7. Solve

In [ ]:
import cProfile
import pstats

profiler = cProfile.Profile()

profiler.enable()

frames = solver.solve()

profiler.disable()

stats = pstats.Stats(profiler)
stats.sort_stats("cumtime").print_stats(30)

## 8. Visualization

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

# Visualize the magnitude (amplitude) of the complex field
ani = solver.animate(
    component='abs',      # 'abs' shows the amplitude |A|
    overlay=None,
    mode='surface',       # 'surface' shows the pattern amplitude clearly
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

# Visualize the phase (angle) of the complex field
ani_phase = solver.animate(
    component='angle',    # 'angle' shows the arg(A)
    overlay=None,
    mode='surface',       # 'surface' shows the pattern amplitude clearly
    physical=True
)

HTML(ani_phase.to_jshtml())

In [ ]:
ani.save('ginzburg_landau_cgle.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to ginzburg_landau_cgle.mp4")

print("\n" + "="*60)
print("TRY THESE PARAMETER COMBINATIONS:")
print("="*60)
print("\n1. Spiral Waves (stable defects):")
print("   ALPHA = 0.5, BETA = -1.0")
print("   → Rotating spiral patterns with stable cores")
print("\n2. Defect Turbulence (Benjamin-Feir instability):")
print("   ALPHA = 1.5, BETA = -1.5")
print("   → Chaotic creation and annihilation of phase defects")
print("\n3. Plane Waves (stable uniform oscillation):")
print("   ALPHA = 0.0, BETA = 0.0")
print("   → Uniform amplitude, simple phase gradient")
print("="*60)